# Data Preprocessing for Hotel Review Sentiment Analysis

# 1. TripAdvisor Hotel Review Dataset Preprocessing

In [70]:
import pandas as pd
import numpy as np
import re
import os

In [71]:
# Load the TripAdvisor hotel reviews dataset
df_tripadvisor = pd.read_csv('../../tripadvisor_hotel_reviews.csv')

# Display basic information
print("Dataset shape:", df_tripadvisor.shape)
print("Columns:", df_tripadvisor.columns.tolist())
print("\nFirst 5 rows:")
display(df_tripadvisor.head())
print("\nData info:")
df_tripadvisor.info()
print("\nMissing values:")
print(df_tripadvisor.isnull().sum())
print("\nDuplicated rows:", df_tripadvisor.duplicated().sum())

# If the dataset does not contain Review and Rating columns, the column list above can help adjust the pipeline.

Dataset shape: (20491, 2)
Columns: ['Review', 'Rating']

First 5 rows:


,Review,Rating
0,nice hotel expensive parking got good deal sta...,4
1,ok nothing special charge diamond member hilto...,2
2,nice rooms not 4* experience hotel monaco seat...,3
3,"unique, great stay, wonderful time hotel monac...",5
4,"great stay great stay, went seahawk game aweso...",5



Data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20491 entries, 0 to 20490
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Review  20491 non-null  object
 1   Rating  20491 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 320.3+ KB

Missing values:
Review    0
Rating    0
dtype: int64

Duplicated rows: 0


In [72]:
# Keep only useful columns: review text and rating
expected_columns = ['Review', 'Rating']
if not all(col in df_tripadvisor.columns for col in expected_columns):
    raise ValueError(f"Expected columns not found. Dataset columns: {df_tripadvisor.columns.tolist()}")

df_tripadvisor = df_tripadvisor[expected_columns].copy()

# Drop rows with missing review text or rating
df_tripadvisor = df_tripadvisor.dropna(subset=['Review', 'Rating'])

# Drop duplicate reviews
df_tripadvisor = df_tripadvisor.drop_duplicates(subset=['Review'])

print("After cleaning - shape:", df_tripadvisor.shape)

After cleaning - shape: (20491, 2)


In [73]:
# Define clean_text function
def clean_text(text):
    # Convert to string
    text = str(text)
    # Lowercase
    text = text.lower()
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove non-English letters and punctuation
    text = re.sub(r'[^a-z\s]', '', text)
    # Normalise whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning
df_tripadvisor['clean_review'] = df_tripadvisor['Review'].apply(clean_text)

# Remove rows where clean_review is empty
df_tripadvisor = df_tripadvisor[df_tripadvisor['clean_review'].str.len() > 0]

print("After text cleaning - shape:", df_tripadvisor.shape)

After text cleaning - shape: (20491, 3)


In [74]:
# Create sentiment label from rating
def rating_to_sentiment(rating):
    if rating >= 4:
        return 'positive'
    elif rating == 3:
        return 'neutral'
    else:
        return 'negative'

df_tripadvisor['sentiment'] = df_tripadvisor['Rating'].apply(rating_to_sentiment)

# Add token_count for EDA
df_tripadvisor['token_count'] = df_tripadvisor['clean_review'].str.split().apply(len)

# Print sentiment distribution
print("Sentiment distribution:")
print(df_tripadvisor['sentiment'].value_counts())
print("\nAverage token count by sentiment:")
print(df_tripadvisor.groupby('sentiment')['token_count'].mean())

Sentiment distribution:
sentiment
positive    15093
negative     3214
neutral      2184
Name: count, dtype: int64

Average token count by sentiment:
sentiment
negative    117.276914
neutral     111.810440
positive     97.324654
Name: token_count, dtype: float64


In [75]:
# Save cleaned data
os.makedirs('../data/processed', exist_ok=True)
df_tripadvisor.to_csv('../data/processed/tripadvisor_cleaned.csv', index=False)
print("Saved cleaned TripAdvisor data to ../data/processed/tripadvisor_cleaned.csv")

Saved cleaned TripAdvisor data to ../data/processed/tripadvisor_cleaned.csv


# 2. TA13 Hotel Review Dataset Preprocessing

In [76]:
# Load the TA13 hotel reviews dataset
df_ta13 = pd.read_csv('../../TA13.csv')

# Display basic information
print("Dataset shape:", df_ta13.shape)
print("Columns:", df_ta13.columns.tolist())
print("\nFirst 5 rows:")
display(df_ta13.head())
print("\nData info:")
df_ta13.info()
print("\nMissing values:")
print(df_ta13.isnull().sum())
print("\nDuplicated rows:", df_ta13.duplicated().sum())

Dataset shape: (80122, 10)
Columns: ['offering_id', 'user_id', 'overall', 'value', 'service', 'location', 'rooms', 'cleanliness', 'sleep_quality', 'review_text']

First 5 rows:


,offering_id,user_id,overall,value,service,location,rooms,cleanliness,sleep_quality,review_text
0,1762573,FB1032DECE1162CB3556D05F278AAFFD,4.0,4.0,4.0,5.0,4.0,5.0,4.0,“Great Stay” This is a great property in Midto...
1,1762573,BA524A238B1171206691A6CC3F28F266,4.0,3.0,4.0,5.0,5.0,5.0,5.0,“Its the best of the Andaz Brand in the US.......
2,1456560,EC6CB11E9DC8761710DDA3CF48DD995F,4.0,4.0,4.0,5.0,4.0,5.0,5.0,“A Nice Stay for NYC!” This hotel is a nice st...
3,1762573,C81AB7D49D98FA410EA191E15F427BEC,4.0,5.0,4.0,5.0,5.0,5.0,5.0,“Stunningly Wonderful!” Other hotels in NYC th...
4,1456560,2404E3630B78BB9E8D6583076FBA0742,4.0,4.0,4.0,4.0,4.0,4.0,5.0,"“Modern, minimalist, central hotel” We got a r..."



Data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80122 entries, 0 to 80121
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   offering_id    80122 non-null  int64  
 1   user_id        80122 non-null  object 
 2   overall        80122 non-null  float64
 3   value          72861 non-null  float64
 4   service        72494 non-null  float64
 5   location       64545 non-null  float64
 6   rooms          67724 non-null  float64
 7   cleanliness    72979 non-null  float64
 8   sleep_quality  45586 non-null  float64
 9   review_text    80122 non-null  object 
dtypes: float64(7), int64(1), object(2)
memory usage: 6.1+ MB

Missing values:
offering_id          0
user_id              0
overall              0
value             7261
service           7628
location         15577
rooms            12398
cleanliness       7143
sleep_quality    34536
review_text          0
dtype: int64

Duplicated rows: 0


In [77]:
# Keep useful columns
useful_columns = ['offering_id', 'user_id', 'overall', 'value', 'service', 'location', 'rooms', 'cleanliness', 'sleep_quality', 'review_text']
missing_columns = [col for col in useful_columns if col not in df_ta13.columns]
if missing_columns:
    raise ValueError(f"Missing expected TA13 columns: {missing_columns}. Dataset columns: {df_ta13.columns.tolist()}")

df_ta13 = df_ta13[useful_columns].copy()

# Drop rows with missing review_text or overall
df_ta13 = df_ta13.dropna(subset=['review_text', 'overall'])

# Drop duplicate reviews based on review_text
df_ta13 = df_ta13.drop_duplicates(subset=['review_text'])

print("After cleaning - shape:", df_ta13.shape)

After cleaning - shape: (80121, 10)


In [78]:
# Apply the same clean_text function
df_ta13['clean_review'] = df_ta13['review_text'].apply(clean_text)

# Remove rows where clean_review is empty
df_ta13 = df_ta13[df_ta13['clean_review'].str.len() > 0]

print("After text cleaning - shape:", df_ta13.shape)

After text cleaning - shape: (79756, 11)


In [79]:
# Create sentiment label from overall
df_ta13['sentiment'] = df_ta13['overall'].apply(rating_to_sentiment)

# Create aspect sentiment labels
aspect_columns = ['value', 'service', 'location', 'rooms', 'cleanliness', 'sleep_quality']

def aspect_rating_to_sentiment(rating):
    if pd.isna(rating):
        return 'not mentioned'
    elif rating >= 4:
        return 'positive'
    elif rating == 3:
        return 'neutral'
    else:
        return 'negative'

for aspect in aspect_columns:
    df_ta13[f'{aspect}_sentiment'] = df_ta13[aspect].apply(aspect_rating_to_sentiment)

# Add token_count for EDA
df_ta13['token_count'] = df_ta13['clean_review'].str.split().apply(len)

# Print sentiment distribution
print("Overall sentiment distribution:")
print(df_ta13['sentiment'].value_counts())
print("\nAspect sentiment distributions:")
for aspect in aspect_columns:
    print(f"{aspect}:")
    print(df_ta13[f'{aspect}_sentiment'].value_counts())
    print()

Overall sentiment distribution:
sentiment
positive    56958
neutral     15743
negative     7055
Name: count, dtype: int64

Aspect sentiment distributions:
value:
value_sentiment
positive         48035
neutral          16851
negative          7643
not mentioned     7227
Name: count, dtype: int64

service:
service_sentiment
positive         53552
neutral          12338
not mentioned     7594
negative          6272
Name: count, dtype: int64

location:
location_sentiment
positive         53410
not mentioned    15544
neutral           8149
negative          2653
Name: count, dtype: int64

rooms:
rooms_sentiment
positive         46662
neutral          14050
not mentioned    12366
negative          6678
Name: count, dtype: int64

cleanliness:
cleanliness_sentiment
positive         58821
neutral           9500
not mentioned     7107
negative          4328
Name: count, dtype: int64

sleep_quality:
sleep_quality_sentiment
not mentioned    34382
positive         34231
neutral           7560
negat

In [80]:
# Save cleaned data
df_ta13.to_csv('../data/processed/ta13_cleaned.csv', index=False)
print("Saved cleaned TA13 data to ../data/processed/ta13_cleaned.csv")

Saved cleaned TA13 data to ../data/processed/ta13_cleaned.csv
